# Hash-Based Pseudonymization with PAMOLA.CORE

**Goal**: Demonstrate hash-based pseudonymization on Bank Churn dataset (ANON-PSEUDO-001 workaround verification)

**What you'll learn:**
- Configure `HashBasedPseudonymizationOperation` with `sha3_256` + fixed salt
- Verify reproducibility across runs (`use_pepper=False`)
- Verify null preservation (`null_strategy='PRESERVE'`)
- Verify family linkage preservation (same FAMILYID → same pseudonym)
- Document Gap A2: `preserve_format` is not supported

**Workaround doc:** `docs/ba-artifacts/happy_path_op_config_workarounds.md`  
**BA spec:** `docs/ba-artifacts/TITAN_PAMOLA_PROD_HAPPY_PATH_2026-05-17.md`

---

## Step 1: Setup and Imports

Auto-detects PAMOLA installation path (searches up 6 levels).
If not found locally, auto-installs from GitHub.

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time

print("Detecting PAMOLA installation...\n")

# Auto-detect project root (search up 6 levels for pamola_core/)
current_dir = Path.cwd()
project_root = current_dir
pamola_found = False

for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"Found pamola_core at level {level}: {project_root}")
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break
    project_root = project_root.parent

if not pamola_found:
    print("pamola_core not found in project structure")
    print("Attempting to install from GitHub...\n")
    try:
        import subprocess
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"],
            capture_output=True, text=True, check=True
        )
        print("Successfully installed pamola_core from GitHub!")
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Installation failed: {e.stderr}")

try:
    from pamola_core.anonymization.pseudonymization.hash_based_op import HashBasedPseudonymizationOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter

    print("All imports successful!")
    print(f"Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"Project root:  {project_root.name}")
    print(f"Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)

except ImportError as e:
    print(f"Import failed: {e}")
    raise

## Step 2: Load Bank Churn Dataset

Load real Bank Churn dataset from cross-repo absolute path.
Select only columns relevant to pseudonymization: ID, FAMILYID, CardNumber.

In [ ]:
CSV_PATH = "E:/DTX/pamola-production/frontend/pamola-spa/tests/e2e/shared/sample-dataset/S_CHURN_BANK_CANADA_10K.csv"

print(f"Loading dataset from: {CSV_PATH}\n")
df_full = pd.read_csv(CSV_PATH)
df = df_full[["ID", "FAMILYID", "CardNumber"]].copy()

print(f"Total rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nNull counts:")
print(df.isnull().sum())
print(f"\nSample (first 5 rows):")
print(df.head())
print(f"\nFAMILYID unique values: {df['FAMILYID'].nunique()} (excluding null)")
print(f"CardNumber sample values:")
print(df['CardNumber'].head(3).tolist())

## Step 3: Setup Shared Environment

Create DataSource and task infrastructure following the pattern from existing advanced notebooks.

In [ ]:
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'pseudo_output'
os.makedirs(task_dir, exist_ok=True)
print(f"Task directory: {task_dir}")

task_reporter = TaskReporter(
    task_id="pseudo_bank_churn_001",
    task_type="hash_pseudonymization",
    description="Bank Churn pseudonymization workaround verification",
    report_path=task_dir
)
print("TaskReporter initialized")

kwargs = {"dataset_name": "main_dataset"}
data_source = DataSource(dataframes={"main_dataset": df})
print("DataSource created")
print("\nShared environment ready!")

## Step 4: Workaround A1 — Verify Pseudonymization Config (ANON-PSEUDO-001)

Config from workaround doc:
- `algorithm='sha3_256'` (BA wanted hmac_sha256, not supported — sha3_256 is the workaround)
- `salt_config={'source':'parameter','value':WORKSPACE_SALT_HEX}` — fixed salt for reproducibility
- `use_pepper=False` — MUST be False; default True gives random pepper per session (non-reproducible)
- `output_length=16` — truncate to 16 chars
- `null_strategy='PRESERVE'` — keep nulls as-is (for FAMILYID)

**Assertions:**
1. Output length == 16 for non-null values
2. null → null preserved (FAMILYID has nulls)
3. Same input → same output on re-run (reproducibility)
4. Same FAMILYID across rows → same pseudonym (linkage preserved)

In [ ]:
# Fixed salt — same 64 hex chars across all 3 op instances for reproducibility
WORKSPACE_SALT_HEX = "a" * 64  # 32-byte hex; placeholder — use real workspace secret in production

print("=" * 80)
print("WORKAROUND A1: Hash-Based Pseudonymization Config Verification")
print("=" * 80)

# ----- Run 1: Pseudonymize ID field -----
op_id = HashBasedPseudonymizationOperation(
    field_name="ID",
    algorithm="sha3_256",
    salt_config={"source": "parameter", "value": WORKSPACE_SALT_HEX},
    use_pepper=False,
    output_length=16,
    null_strategy="PRESERVE",
)

tracker_id = HierarchicalProgressTracker(total=6, description="Pseudo ID", unit="steps")
start = time.time()
result_id = op_id.execute(
    data_source=data_source,
    task_dir=task_dir / 'pseudo_id',
    reporter=task_reporter,
    progress_tracker=tracker_id,
    **kwargs
)
elapsed_id = time.time() - start
print(f"\nID pseudonymization completed in {elapsed_id:.2f}s")

# Load result
output_files = sorted(
    list((task_dir / 'pseudo_id' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files:
    result_df_id = pd.read_csv(output_files[0])
    print(f"\nResult columns: {list(result_df_id.columns)}")
    print(f"\nSample output (ID → pseudonym):")
    print(result_df_id[["ID"]].head())
else:
    result_df_id = None
    print("WARNING: No output file found")

In [ ]:
# ----- Run FAMILYID with null_strategy=PRESERVE -----
op_fam = HashBasedPseudonymizationOperation(
    field_name="FAMILYID",
    algorithm="sha3_256",
    salt_config={"source": "parameter", "value": WORKSPACE_SALT_HEX},
    use_pepper=False,
    output_length=16,
    null_strategy="PRESERVE",
)

# Run 1
data_source_fam = DataSource(dataframes={"main_dataset": df})
tracker_fam1 = HierarchicalProgressTracker(total=6, description="Pseudo FAMILYID run1", unit="steps")
result_fam1 = op_fam.execute(
    data_source=data_source_fam,
    task_dir=task_dir / 'pseudo_familyid_run1',
    reporter=task_reporter,
    progress_tracker=tracker_fam1,
    **kwargs
)
out1_files = sorted(
    list((task_dir / 'pseudo_familyid_run1' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
result_fam_df1 = pd.read_csv(out1_files[0]) if out1_files else None

# Run 2 — same op, same data, expect identical output
data_source_fam2 = DataSource(dataframes={"main_dataset": df})
tracker_fam2 = HierarchicalProgressTracker(total=6, description="Pseudo FAMILYID run2", unit="steps")
result_fam2 = op_fam.execute(
    data_source=data_source_fam2,
    task_dir=task_dir / 'pseudo_familyid_run2',
    reporter=task_reporter,
    progress_tracker=tracker_fam2,
    **kwargs
)
out2_files = sorted(
    list((task_dir / 'pseudo_familyid_run2' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
result_fam_df2 = pd.read_csv(out2_files[0]) if out2_files else None

print("=" * 80)
print("ASSERTIONS")
print("=" * 80)

if result_fam_df1 is not None and result_fam_df2 is not None:
    fam_col = "FAMILYID"  # REPLACE mode overwrites the column in-place

    # --- Assert 1: null → null preserved ---
    original_nulls = df["FAMILYID"].isnull().sum()
    result_nulls = result_fam_df1[fam_col].isnull().sum()
    null_preserved = original_nulls == result_nulls
    print(f"[{'PASS' if null_preserved else 'FAIL'}] Null preservation:")
    print(f"   Original nulls: {original_nulls}, Result nulls: {result_nulls}")

    # --- Assert 2: Reproducibility (run1 == run2) ---
    reproducible = result_fam_df1[fam_col].equals(result_fam_df2[fam_col])
    print(f"[{'PASS' if reproducible else 'FAIL'}] Reproducibility (run1 == run2):")
    if not reproducible:
        diff_count = (result_fam_df1[fam_col] != result_fam_df2[fam_col]).sum()
        print(f"   Differing rows: {diff_count}")

    # --- Assert 3: Output length == 16 for non-null ---
    non_null_mask = result_fam_df1[fam_col].notna()
    lengths = result_fam_df1.loc[non_null_mask, fam_col].astype(str).str.len()
    all_16 = (lengths == 16).all()
    print(f"[{'PASS' if all_16 else 'FAIL'}] Output length == 16 for non-null values:")
    print(f"   Length distribution: {lengths.value_counts().to_dict()}")

    # --- Assert 4: Linkage — same FAMILYID → same pseudonym ---
    # Find rows that share the same FAMILYID in original
    original_fam = df["FAMILYID"].dropna()
    # Pick a FAMILYID value that appears multiple times
    fam_counts = original_fam.value_counts()
    multi_fam = fam_counts[fam_counts > 1]
    if len(multi_fam) > 0:
        test_fam_val = multi_fam.index[0]
        test_indices = df[df["FAMILYID"] == test_fam_val].index
        pseudo_values = result_fam_df1.loc[test_indices, fam_col].unique()
        linkage_ok = len(pseudo_values) == 1
        print(f"[{'PASS' if linkage_ok else 'FAIL'}] Linkage preservation:")
        print(f"   FAMILYID '{test_fam_val}' appears {len(test_indices)} times → {len(pseudo_values)} unique pseudonym(s): {pseudo_values}")
    else:
        print("[SKIP] Linkage test: No repeated FAMILYID values found in dataset")

    # Sample output
    print(f"\nSample FAMILYID pseudonymization (first 10 rows):")
    sample_cols = ["FAMILYID"]
    print(result_fam_df1[sample_cols].head(10))
else:
    print("BLOCKED: Output files not found — cannot run assertions")

## Step 5: Gap A2 — preserve_format Not Supported (ANON-PSEUDO-001 CardNumber)

BA spec wants `preserve_format: true` for CardNumber so output looks like `XXXX XXXXX XXXX XXXX`.

**Finding:** `hash_based_op.py:228` — `output_format: str = 'hex'`, no format-preservation logic.
All output is flat hex string. The 'T' character and spacing in the original are lost.

**Workaround chosen:** Option A (accept plain 16-char hex, note as known deviation).

In [ ]:
print("=" * 80)
print("GAP A2: preserve_format NOT SUPPORTED — CardNumber format demonstration")
print("=" * 80)

# Sample 5 real CardNumber values from dataset
sample_cards = df["CardNumber"].dropna().head(5).tolist()
print(f"Real CardNumber sample values (note 'T' in group 2):")
for c in sample_cards:
    print(f"  Input:  {repr(c)}")

# Run pseudonymization on CardNumber
df_card = df[["CardNumber"]].copy()
op_card = HashBasedPseudonymizationOperation(
    field_name="CardNumber",
    algorithm="sha3_256",
    salt_config={"source": "parameter", "value": WORKSPACE_SALT_HEX},
    use_pepper=False,
    output_length=16,
    null_strategy="PRESERVE",
)

data_source_card = DataSource(dataframes={"main_dataset": df_card})
tracker_card = HierarchicalProgressTracker(total=6, description="Pseudo CardNumber", unit="steps")
result_card = op_card.execute(
    data_source=data_source_card,
    task_dir=task_dir / 'pseudo_cardnumber',
    reporter=task_reporter,
    progress_tracker=tracker_card,
    dataset_name="main_dataset"
)

card_out_files = sorted(
    list((task_dir / 'pseudo_cardnumber' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if card_out_files:
    result_card_df = pd.read_csv(card_out_files[0])
    print(f"\nCardNumber pseudonymization output (first 5):")
    print(result_card_df["CardNumber"].head(5).tolist())

    # Check that 'T' and spaces are gone
    output_values = result_card_df["CardNumber"].dropna().astype(str)
    has_T = output_values.str.contains('T').any()
    has_space = output_values.str.contains(' ').any()
    print(f"\n[GAP CONFIRMED] Output contains 'T': {has_T} (expected False — format lost)")
    print(f"[GAP CONFIRMED] Output contains spaces: {has_space} (expected False — format lost)")
    print(f"\nKnown gap (workaround doc Option A): visual format 'XXXX XXXXX XXXX XXXX' is lost.")
    print(f"Output is plain 16-char hex string. Source: hash_based_op.py:228 — output_format='hex', no format-preservation logic.")
else:
    print("WARNING: No output file found for CardNumber")

## Step 6: Bank Churn Workaround Verification Summary (ANON-PSEUDO-001)

| Claim | Status | Evidence |
|---|---|---|
| `sha3_256` + fixed salt + `use_pepper=False` → reproducible | See assertion above | run1 == run2 check |
| `null_strategy='PRESERVE'` → FAMILYID nulls kept | See assertion above | null count before/after |
| Same FAMILYID → same pseudonym (linkage) | See assertion above | multi-row FAMILYID check |
| Gap A2: `preserve_format` not supported | CONFIRMED | CardNumber loses 'T' and spacing |

**Source refs:**
- `hash_based_op.py:148` — algorithm enum: `["sha3_256", "sha3_512"]` only
- `hash_based_op.py:222` — `use_pepper: bool = True` default; `:923-925` random pepper per session  
- `hash_based_op.py:226` — `null_strategy: str = 'PRESERVE'` default  
- `hash_based_op.py:228` — `output_format: str = 'hex'`, no format-preservation logic